# 🔮 Bước 4: Predictive Analytics
Dự đoán Purchase Amount (USD)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib, os

CHARTS = '../output/charts'
MODELS = '../output/models'
os.makedirs(MODELS, exist_ok=True)

try:
    df = pd.read_pickle('../output/df_clean.pkl')
except:
    df = pd.read_csv('../data/shopping_trends.csv')
    for col in ['Gender','Category','Season','Size','Shipping Type','Subscription Status','Discount Applied','Promo Code Used','Frequency of Purchases']:
        df[col] = df[col].astype('category')

print("✅ Dữ liệu:", df.shape)


## 4.1 Feature Engineering

In [ ]:
df_fe = df.copy()

# Encode binary
for col, mapping in [('Subscription Status', {'Yes':1,'No':0}),
                     ('Discount Applied',     {'Yes':1,'No':0}),
                     ('Promo Code Used',       {'Yes':1,'No':0})]:
    df_fe[col+'_bin'] = df_fe[col].astype(str).map(mapping).fillna(0).astype(int)

# Encode frequency
freq_map = {'Weekly':52,'Bi-Weekly':26,'Fortnightly':26,
            'Monthly':12,'Quarterly':4,'Annually':1,'Every 3 Months':4}
df_fe['Frequency_num'] = df_fe['Frequency of Purchases'].astype(str).map(freq_map).fillna(4)

# One-Hot Encoding
for col in ['Gender','Category','Season','Size','Shipping Type']:
    dummies = pd.get_dummies(df_fe[col].astype(str), prefix=col, drop_first=True)
    df_fe = pd.concat([df_fe, dummies], axis=1)

print("✅ Feature Engineering xong!")
df_fe.shape


## 4.2 Chuẩn bị X, y & Train/Test Split

In [ ]:
target = 'Purchase Amount (USD)'
drop_cols = ['Customer ID','Item Purchased','Color','Location','Payment Method',
             'Preferred Payment Method','Gender','Category','Season','Size',
             'Shipping Type','Subscription Status','Discount Applied','Promo Code Used',
             'Frequency of Purchases','Age Group', target]
drop_cols = [c for c in drop_cols if c in df_fe.columns]

feature_cols = [c for c in df_fe.columns
                if c not in drop_cols and df_fe[c].dtype in [np.int64, np.float64, bool, np.uint8]]

X = df_fe[feature_cols].fillna(0)
y = df_fe[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Features   : {X.shape[1]} biến → {list(X.columns)}")
print(f"Train size : {X_train.shape[0]:,} mẫu")
print(f"Test size  : {X_test.shape[0]:,} mẫu")


## 4.3 Huấn luyện & Đánh giá mô hình

In [ ]:
results = []

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)})
    print(f"  [{name}]  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}")
    return y_pred

# Linear Regression
print("\n🔷 Linear Regression")
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = evaluate('Linear Regression', y_test, lr.predict(X_test))
joblib.dump(lr, f'{MODELS}/lr.pkl')

# Random Forest
print("\n🌳 Random Forest")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = evaluate('Random Forest', y_test, rf.predict(X_test))
joblib.dump(rf, f'{MODELS}/rf.pkl')

# XGBoost
try:
    from xgboost import XGBRegressor
    print("\n⚡ XGBoost")
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5,
                        random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    pred_xgb = evaluate('XGBoost', y_test, xgb.predict(X_test))
    joblib.dump(xgb, f'{MODELS}/xgb.pkl')
except ImportError:
    print("  XGBoost không khả dụng, bỏ qua.")

metrics_df = pd.DataFrame(results)
print()
display(metrics_df)


## 4.4 Trực quan hóa kết quả

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# So sánh mô hình
axes[0].bar(metrics_df['Model'], metrics_df['MAE'],
            color=sns.color_palette('Set2', len(metrics_df)))
axes[0].set_title('MAE (thấp hơn = tốt hơn)')
axes[0].set_ylabel('MAE')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=15)

axes[1].bar(metrics_df['Model'], metrics_df['RMSE'],
            color=sns.color_palette('Set2', len(metrics_df)))
axes[1].set_title('RMSE (thấp hơn = tốt hơn)')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=15)

axes[2].bar(metrics_df['Model'], metrics_df['R2'],
            color=sns.color_palette('Set2', len(metrics_df)))
axes[2].set_title('R² Score (cao hơn = tốt hơn)')
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=15)

plt.suptitle('So Sánh Hiệu Suất Các Mô Hình', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_10_model_compare.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Actual vs Predicted (Linear Regression)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, pred_lr, alpha=0.4, color='steelblue', s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title('Linear Regression: Actual vs Predicted')

# Feature Importance (Random Forest)
importance = pd.Series(rf.feature_importances_, index=feature_cols).nlargest(12)
axes[1].barh(importance.index[::-1], importance.values[::-1],
             color=sns.color_palette('viridis', 12))
axes[1].set_title('Random Forest: Feature Importance')

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_11_pred_importance.png', dpi=150, bbox_inches='tight')
plt.show()
